# 01 — Population Reconciliation

**Goal.** Build a single authoritative annual population series for each New
York county and year (2000–present) from multiple overlapping sources, with
clear provenance.

## Sources

| Source | Years | Geography | Kind |
|---|---|---|---|
| Census PEP 2000–2010 intercensal | 2000–2010 | NY counties | totals only |
| Census PEP 2010–2020 intercensal | 2010–2020 | NY counties | totals + components + rates |
| Census PEP 2020+ postcensal | 2020–2025 | NY counties | totals + components + rates |
| NYSDOL annual estimates | 1970–2025 | NY + counties | totals only |

## Reconciliation rules

Every retained value is a **July 1** estimate. We deliberately do not anchor
on the April 1 decennial enumerations at 2000/2010/2020 — using April 1
counts inside an otherwise July 1 series introduces a ~3-month phase shift
at each decade boundary that distorts trend visualizations and confuses
year-over-year change.

1. **2000–2019** — **NYSDOL July 1 intercensal estimate**. NYSDOL publishes
   an annual July 1 series back to 1970 with consistent methodology, and
   the legacy R workflow used it as the authoritative intercensal source.
   The series flows continuously through the 2000 and 2010 decennials.
2. **2020+** — **Census PEP July 1 postcensal estimate**, latest vintage.
   The Census Bureau anchors PEP's July 1, 2020 value to the April 1, 2020
   decennial count and then carries it forward year by year — so 2020+ is
   PEP's natural domain and we follow that.
3. **Vintage overlap** — when two PEP files cover the same year, we keep
   the later vintage because the Census Bureau revised its methodology in
   the newer release.
4. **Caveat — components of change do NOT reconcile across the decennial
   seam.** The Census Bureau smooths its intercensal July 1 totals so they
   land on the new decennial count at each decade boundary, but the Bureau's
   published components (births, deaths, migration) for those same years
   still sum to the original (unsmoothed) postcensal totals. We carry both
   and flag the difference as the "intercensal residual" rather than
   adjusting the components ourselves.

## Output

- `data_interim/population_all_sources.parquet` — stacked long-format raw series for QA
- `data_interim/population_reconciled.parquet` — single authoritative value per `(geoid, year)`

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from popfc.data.census import load_all_pep
from popfc.data.nysdol import load_nysdol_annual
from popfc.paths import DATA_INTERIM, FULL_FIPS
from popfc.reconcile import reconcile_county_population, resolve_pep_vintage

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

WASHINGTON = FULL_FIPS  # '36115'

# Five demographic neighbors used for Phase-1 validation cohort.
NEIGHBORS = {
    "36091": "Saratoga",
    "36113": "Warren",
    "36083": "Rensselaer",
    "36031": "Essex",
    "36021": "Columbia",
}
COHORT = {WASHINGTON: "Washington", **NEIGHBORS}

## 1. Load all raw sources

Each loader emits the canonical long-format `POP_LONG_COLUMNS` schema so we
can stack without any source-specific cleanup here.

In [ ]:
pep = load_all_pep(state_filter="36")
pop_pep = pep["population"]
comp_pep = pep["components"]

pop_nysdol = load_nysdol_annual()

# Stack population frames only (components come from Census alone).
# Coerce population to a common nullable Int64 dtype. NYSDOL already emits
# Int64; PEP emits object-dtype (the 2010-2020 CENSUS2010POP column has
# mixed int/str values in the raw CSV). `to_numeric` handles both uniformly.
pop_pep["population"] = pd.to_numeric(
    pop_pep["population"], errors="coerce"
).astype("Int64")
pop_nysdol["population"] = pop_nysdol["population"].astype("Int64")
pop_all = pd.concat([pop_pep, pop_nysdol], ignore_index=True)

print(f"pop_pep:     {len(pop_pep):>6,} rows  "
      f"(vintages: {sorted(pop_pep['vintage'].unique())})")
print(f"pop_nysdol:  {len(pop_nysdol):>6,} rows  "
      f"(vintage:  {pop_nysdol['vintage'].unique().tolist()})")
print(f"comp_pep:    {len(comp_pep):>6,} rows")
print(f"pop_all:     {len(pop_all):>6,} rows (stacked)")

## 2. Source inventory

What years, kinds, and sources are actually covered?

In [ ]:
inventory = (
    pop_all.groupby(["source", "vintage", "kind"])["year"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "year_min", "max": "year_max", "count": "n_rows"})
    .reset_index()
    .sort_values(["source", "year_min", "kind"])
)
inventory

## 3. Washington County across all sources

Pivot every source × kind into a single table so we can see the disagreements
directly. Column headers encode `(source | kind | vintage)`.

In [ ]:
def pivot_county(df: pd.DataFrame, geoid: str) -> pd.DataFrame:
    sub = df[df["geoid"] == geoid].copy()
    sub["col"] = (
        sub["source"] + " | " + sub["kind"] + " | " + sub["vintage"]
    )
    wide = (
        sub.pivot_table(
            index="year", columns="col", values="population", aggfunc="first"
        )
        .sort_index()
    )
    return wide

wash_wide = pivot_county(pop_all, WASHINGTON)
wash_wide.tail(15)

### Where do sources disagree?

For each year that has ≥ 2 values, show the spread (max − min) and range.

In [ ]:
def spread(df_wide: pd.DataFrame) -> pd.DataFrame:
    stats = pd.DataFrame(index=df_wide.index)
    stats["n_values"] = df_wide.notna().sum(axis=1)
    stats["min"] = df_wide.min(axis=1)
    stats["max"] = df_wide.max(axis=1)
    stats["spread"] = stats["max"] - stats["min"]
    return stats[stats["n_values"] >= 2].sort_values("spread", ascending=False)

spread(wash_wide).head(20)

## 4. Visualize disagreement — Washington + 5 neighbors

One subplot per county. Each line is a (source, kind) combination.

In [ ]:
def plot_county_series(df: pd.DataFrame, geoid: str, name: str, ax) -> None:
    sub = df[df["geoid"] == geoid].copy()
    sub["series"] = sub["source"] + "/" + sub["kind"]
    for series_name, g in sub.groupby("series"):
        g = g.sort_values("year")
        ax.plot(g["year"], g["population"], marker="o", markersize=3,
                linewidth=1, label=series_name)
    ax.set_title(f"{name} ({geoid})")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("year")
    ax.set_ylabel("population")

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
for ax, (geoid, name) in zip(axes.flat, COHORT.items()):
    plot_county_series(pop_all, geoid, name, ax)
# Single legend outside
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8,
           bbox_to_anchor=(0.5, -0.03))
fig.suptitle("Population series by source/kind — Washington + neighbors",
             fontsize=13)
fig.tight_layout()
plt.show()

## 4b. Outlier audit — statewide source disagreements

Where do the available sources disagree by more than expected? For every
NY county × year with ≥ 2 source values covering that year, compute the
spread (max − min) and the relative spread as a fraction of the max
value. A relative spread > 0.5% (~300 people in a 60,000-pop county)
is flagged — usually it indicates either a Census vintage revision or
a NYSDOL methodology change.

This is a data-quality check, not a forecast input. The reconciliation
rule (next section) picks one source per row, so the spreads here are
informational. But if a county has wide spreads year after year, that's
worth knowing before we trust its forecast.

In [ ]:
def spread_all(pop_all_df: pd.DataFrame, county_only: bool = True) -> pd.DataFrame:
    df = pop_all_df.copy()
    if county_only:
        df = df[df["county_fips"] != "000"]
    grouped = (
        df.dropna(subset=["population"])
          .groupby(["geoid", "geography", "year"])["population"]
          .agg(["count", "min", "max"])
          .reset_index()
          .rename(columns={"count": "n_values"})
    )
    grouped = grouped[grouped["n_values"] >= 2].copy()
    grouped["spread"] = grouped["max"] - grouped["min"]
    grouped["rel_spread_pct"] = 100.0 * grouped["spread"] / grouped["max"]
    return grouped

spread_state = spread_all(pop_all)
THRESH_PCT = 0.5
flagged = spread_state[spread_state["rel_spread_pct"] > THRESH_PCT].copy()
print(f"Statewide source-disagreement audit:")
print(f"  county-years with >=2 source values: {len(spread_state):>5,}")
print(f"  flagged (rel spread > {THRESH_PCT}% of max): {len(flagged):>5,}  "
      f"({100*len(flagged)/len(spread_state):.1f}%)")
print()
print(f"Top 20 worst by relative spread:")
print(flagged.sort_values("rel_spread_pct", ascending=False)
            .head(20)[["geography", "year", "n_values", "min", "max", "spread", "rel_spread_pct"]]
            .to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

In [ ]:
# Distribution + per-county count of flagged years.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(spread_state["rel_spread_pct"].clip(upper=5.0), bins=50, color="C0", alpha=0.8)
axes[0].axvline(THRESH_PCT, color="C3", linewidth=1.2, linestyle="--",
                label=f"flag threshold ({THRESH_PCT}%)")
axes[0].set_xlabel("relative spread (max - min) / max, %")
axes[0].set_ylabel("count of county-years")
axes[0].set_title("Distribution of source disagreement across NY counties × years")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-county flagged count.
per_county = (
    flagged.groupby("geography").size().sort_values(ascending=False).head(15)
)
axes[1].barh(per_county.index[::-1], per_county.values[::-1], color="C3", alpha=0.7)
axes[1].set_xlabel(f"# years flagged (rel spread > {THRESH_PCT}%)")
axes[1].set_title("Counties with the most flagged years (top 15)")
axes[1].grid(True, alpha=0.3, axis="x")

fig.tight_layout()
plt.show()

# Surface Washington + cohort counties specifically.
cohort_flagged = flagged[flagged["geoid"].isin(COHORT)]
if not cohort_flagged.empty:
    print(f"\nFlagged years in the cohort (Washington + 5 neighbors):")
    print(cohort_flagged.sort_values(["geography", "year"])
                       [["geography", "year", "n_values", "min", "max", "rel_spread_pct"]]
                       .to_string(index=False, float_format=lambda x: f'{x:,.2f}'))
else:
    print(f"\nNo cohort-county years flagged at the {THRESH_PCT}% threshold.")

## 5. Handle PEP vintage overlap

The three Census PEP files overlap at decennial years (e.g., 2020 appears in
both the 2010–2020 intercensal file and the 2020+ postcensal file). We keep
the **later vintage** for non-decennial overlap, and keep both kinds
(`census` vs `estimate`) for decennial years since they carry distinct
semantic meaning.

In [ ]:
# Implementation lives in popfc.reconcile (DEFAULT_PEP_VINTAGE_RANK there).
pop_pep_resolved = resolve_pep_vintage(pop_pep)
print(f"pop_pep:           {len(pop_pep):>6,} rows")
print(f"pop_pep_resolved:  {len(pop_pep_resolved):>6,} rows")
print(f"dropped (older vintage duplicates): "
      f"{len(pop_pep) - len(pop_pep_resolved):,}")

# Sanity: no remaining duplicates on (geoid, year, kind)
dup = pop_pep_resolved.groupby(["geoid", "year", "kind"]).size()
assert (dup == 1).all(), "Unexpected residual duplicates after vintage resolution"
print("OK — no residual duplicates on (geoid, year, kind).")

## 6. Build the reconciled series

Apply the rules from the opening section in priority order. Each row in the
output carries the `source`/`kind`/`vintage` of the selected value, plus a
`rule` column documenting *why* it was chosen.

In [ ]:
# Reconciliation rules live in popfc.reconcile.reconcile_county_population.
reconciled = reconcile_county_population(pop_pep_resolved, pop_nysdol)
print(f"reconciled rows: {len(reconciled):,}")
print("\nRule usage:")
print(reconciled["rule"].value_counts().to_string())
print("\nYear coverage:",
      f"{reconciled['year'].min()} – {reconciled['year'].max()}")

### Reconciled series — Washington County

In [ ]:
reconciled[reconciled["geoid"] == WASHINGTON][
    ["year", "population", "source", "kind", "vintage", "rule"]
].sort_values("year").to_string(index=False)

## 7. QA checks

Verify that the reconciled series satisfies basic sanity constraints:

- one row per `(geoid, year)`
- all 62 NY counties present in every year we claim to cover
- no gaps in the year range per county
- population strictly positive and plausible

In [ ]:
def qa_checks(df: pd.DataFrame) -> None:
    # 1. Unique on (geoid, year)
    dup = df.groupby(["geoid", "year"]).size()
    assert (dup == 1).all(), f"Duplicate (geoid, year) rows: {dup[dup > 1]}"

    # 2. All NY counties present every year. NY has 62 counties + 1 state row.
    # State row uses county_fips '000' -> geoid '36000'.
    counties = df[df["county_fips"] != "000"]
    n_counties_per_year = counties.groupby("year")["geoid"].nunique()
    # Some early years might be state-only in some sources; warn rather than fail.
    bad_years = n_counties_per_year[n_counties_per_year != 62]
    if not bad_years.empty:
        print(f"WARNING: years with != 62 counties:\n{bad_years}")
    else:
        print("OK — all 62 counties present in every year.")

    # 3. No per-county gaps
    def has_gap(g: pd.DataFrame) -> bool:
        years = sorted(g["year"].unique())
        return any(b - a > 1 for a, b in zip(years, years[1:]))
    gaps = counties.groupby("geoid").apply(has_gap, include_groups=False)
    gap_counties = gaps[gaps].index.tolist()
    if gap_counties:
        print(f"WARNING: counties with year gaps: {gap_counties}")
    else:
        print("OK — no per-county year gaps.")

    # 4. Sane populations (positive, ≤ 25M — NY state total ~20M).
    bad_pop = df[(df["population"] <= 0) | (df["population"] > 25_000_000)]
    assert bad_pop.empty, f"Implausible populations:\n{bad_pop}"
    print(f"OK — all populations in (0, 25M]; min={df['population'].min():,}, "
          f"max={df['population'].max():,}")

qa_checks(reconciled)

## 8. Save outputs

Two parquet files into `data_interim/`:

- `population_all_sources.parquet` — stacked raw series for reference / QA
- `population_reconciled.parquet` — the authoritative series

In [ ]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

all_sources_path = DATA_INTERIM / "population_all_sources.parquet"
reconciled_path = DATA_INTERIM / "population_reconciled.parquet"

pop_all.to_parquet(all_sources_path, index=False)
reconciled.to_parquet(reconciled_path, index=False)

print(f"wrote {all_sources_path}  ({len(pop_all):,} rows)")
print(f"wrote {reconciled_path}  ({len(reconciled):,} rows)")

## 9. Benchmark overlay — Cornell PAD projection (Washington)

The reconciled Census/NYSDOL series is the *history*. Cornell PAD's
`padprojections115.xls` is the *projection benchmark* this project's
forecast will eventually be compared against. Show them on the same
chart so we can see (a) how PAD lined up with subsequent observed
history, and (b) where PAD projects Washington's population over the
2025–2040 horizon.

In [ ]:
from popfc.data.cornell import load_cornell_pad

pad = load_cornell_pad()
pad_totals = pad["totals"]

wash_recon = reconciled[reconciled["geoid"] == WASHINGTON].sort_values("year")
wash_pad = pad_totals[pad_totals["geoid"] == WASHINGTON].sort_values("year")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(wash_recon["year"], wash_recon["population"],
        marker="o", markersize=3, linewidth=1.4,
        label="Reconciled (Census/NYSDOL)")
ax.plot(wash_pad["year"], wash_pad["population"],
        marker="s", markersize=3, linewidth=1.2, linestyle="--",
        label=f"Cornell PAD ({wash_pad['vintage'].iloc[0]})")
ax.axvline(2025.5, color="grey", linewidth=0.8, alpha=0.5)
ax.text(2025.7, ax.get_ylim()[1], "→ projection horizon",
        va="top", ha="left", fontsize=9, color="grey")
ax.set_title("Washington County (36115): history vs Cornell PAD projection")
ax.set_xlabel("year")
ax.set_ylabel("population")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

# Side-by-side table for the overlap window 2015–2025.
overlap = pd.merge(
    wash_recon[["year", "population"]].rename(columns={"population": "reconciled"}),
    wash_pad[["year", "population"]].rename(columns={"population": "cornell_pad"}),
    on="year", how="inner",
)
overlap["pad_minus_recon"] = (
    overlap["cornell_pad"].astype("Int64") - overlap["reconciled"].astype("Int64")
)
overlap["pct_diff"] = 100.0 * overlap["pad_minus_recon"].astype("Float64") / overlap["reconciled"].astype("Float64")
print("PAD vs reconciled, overlap years (Washington):")
print(overlap.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

## Next steps (Phase 1 continued)

- **CDC Bridged-Race loader** — done (`src/popfc/data/cdc.py`).
- **NYSDOH population loader** — done (`src/popfc/data/nysdoh.py`).
- **NYSDOH vital statistics (births/deaths)** — deferred to a follow-on
  API pull (see GitHub issue).
- **Notebook 03 — age/sex audit** (CDC Bridged-Race 1990–2020 vs Census
  SYA 2020–2024, continuity across the 2020 seam).
- **Investigate the decennial seam residual** per county — how big is the
  implied mismatch between intercensal smoothing and component sums?
- **Reconciliation logic** promoted to `src/popfc/reconcile.py` — done.